# Explore the Workspace and Compute

Read the configured Azure ML workspace and compute, then inspect representative models, environments, data assets, endpoints, jobs, and pipelines. This notebook performs no mutation.

**Source:** Adapted from the local Azure ML SDK v2 notebook patterns documented in `workshop/SOURCES.md`.

In [ ]:
from itertools import islice
from pathlib import Path
import os

import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
TENANT_ID = os.getenv("AZURE_TENANT_ID", "")
RESOURCE_GROUP = os.environ["AZURE_RESOURCE_GROUP"]
WORKSPACE_NAME = os.environ["AZUREML_WORKSPACE_NAME"]
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
credential = AzureCliCredential(tenant_id=TENANT_ID or None)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

In [ ]:
workspace = ml_client.workspaces.get(WORKSPACE_NAME)
compute = ml_client.compute.get(COMPUTE_NAME)

print(f"Workspace: {workspace.name}")
print(f"Location: {workspace.location}")
print(f"Compute: {compute.name}")
print(f"Compute type: {compute.type}")
print(f"Compute state: {getattr(compute, 'state', 'not reported')}")

collections = {
    "data": ml_client.data.list(),
    "model": ml_client.models.list(),
    "environment": ml_client.environments.list(),
    "endpoint": ml_client.online_endpoints.list(),
    "job": ml_client.jobs.list(),
}
rows = [
    {"asset_type": asset_type, "name": item.name}
    for asset_type, items in collections.items()
    for item in islice(items, 5)
]
asset_inventory = pd.DataFrame(rows)
display(asset_inventory)

assert workspace.name == WORKSPACE_NAME
assert compute.name == COMPUTE_NAME
print("Read-only workspace tour complete.")

## Expected Result

The configured workspace and compute are available, and a small cross-section of predeployed assets is visible. No resources are changed.

Next: `02_register_data_asset.ipynb`.